In [441]:
import re
from pathlib import Path
import polars
import wikipedia
import requests
import mysql.connector
import json

In [442]:
# page_id → categorylinks.cl_from → cl_target_id → linktarget.lt_id → lt_title (category name)

# 0. Stuff I'll delete 

In [443]:
path_target_cats = "/Users/veronicaperez/Documents/GitHub/wikipedia_categorization/data/output/target_categories.json"

In [444]:
with open(path_target_cats,'r') as f:
    target_cats = set(json.load(f))

# 1. BFS code

In [445]:
page_id_smartphone = 167079
page_id = page_id_smartphone
# title = page.title

In [446]:
# connect to local mariadb
def get_connection():
    return mysql.connector.connect(
        host="localhost",
        user="root",
        password="NiallerKFC",
        database="wikipedia"
    )

In [447]:
def get_page_title(page_id: int) -> str:
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(
        """
        SELECT page_title FROM page where page_id = %s
        """,
        (page_id,)
    )
    row = cursor.fetchone()
    cursor.close()
    conn.close()
    return row[0].decode('utf-8') if row else None

In [448]:
# from categorylinks get cl_target_id - input page id
# filter by hidden -> so we don't get the hidden categories
def get_categories_for_page(page_id: int) -> set[str]:
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(
        """
        SELECT lt_title
        FROM categorylinks
        JOIN linktarget ON cl_target_id = lt_id
        LEFT JOIN page ON lt_title = page_title AND page_namespace = 14
        LEFT JOIN page_props ON page_id = pp_page AND pp_propname = 'hiddencat'
        WHERE cl_from = %s
        AND pp_page IS NULL
        """,
        (page_id,)
    )
    results = {row[0].decode('utf-8') for row in cursor.fetchall()} # set so we dont get double
    cursor.close()
    conn.close()
    return results


In [449]:
get_categories_for_page(page_id)

{'Cloud_clients',
 'Consumer_electronics',
 'Information_appliances',
 'Personal_digital_assistants',
 'Smartphones'}

In [450]:
"Cloud_clients".replace('_', ' ')

'Cloud clients'

In [451]:
def get_intersection_cat_target(results_search: set, target_cats = target_cats) -> bool:
    clean_results = {x.replace('_', ' ') for x in results_search}
    
    return clean_results & target_cats

In [452]:
get_intersection_cat_target(get_categories_for_page(page_id), target_cats)

{'Consumer electronics'}

In [453]:
## BFS

page_id_nn = 21523

In [454]:
get_intersection_cat_target(get_categories_for_page(page_id_nn), target_cats)

set()

In [455]:
def find_lt_title_id(lt_title: str) -> int:
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(
        """
        SELECT page_id FROM page WHERE page_title = %s AND page_namespace = 14
        """,
        (lt_title,)
    )
    row = cursor.fetchone()
    cursor.close()
    conn.close()
    return row[0] if row else None

In [456]:
def get_subcats_ids_for_cats(current_cats_titles: set) -> list:
    current_cats_id = [find_lt_title_id(x) for x in current_cats_titles]
    return current_cats_id

In [457]:
# algorithm to find the target categories
def BFS(page_id:int, target_cats) -> list[str]:
    counter = 1
    intersection = dict()
    old_categories = set()
    visited = set() # handle wiki cycles id otn want to check that
    while counter <= 5:
        #first_search 
        if counter == 1:
            # first try we just need the categories for one page
            new_categories = get_categories_for_page(page_id)
            # print(new_categories)
        else: 
            # now we loop over the results from before
            subcats_ids = get_subcats_ids_for_cats(old_categories) # new set of all the subcategories for the new level
            # print(subcats_ids)
            new_categories = set()
            # loop over new pages
            for x in subcats_ids:
                new_categories.update(get_categories_for_page(x))
            # print(new_categories)

        new_intersection = get_intersection_cat_target(new_categories, target_cats) 
        if new_intersection:
            for inter in new_intersection:
                intersection[inter] = intersection.get(inter, 0) + 1
        
        old_categories = new_categories - visited # to check in the next step
        visited.update(new_categories)
        counter +=1

    if not intersection:
        return set()
    
    else:
        max_count = max(intersection.values())
        top = {k for k, v in intersection.items() if v == max_count}
        return top

In [458]:
# algorithm to find the target categories
def BFS_hard(page_id:int, target_cats) -> list[str]:
    """
    This goes up until layer 20
    after layer 15, it asks whether the top is unique
    if it its then exits, if not, it keeps going until there's a winner
    """
    counter = 1
    intersection = dict()
    old_categories = set()
    visited = set() # handle wiki cycles id otn want to check that
    UNIQUE_TOP = False
    while counter <= 10 and not UNIQUE_TOP:
        if counter % 5 ==0:
            print(f"Checking layer {counter}")
            print(f"current dictionary {intersection}")
        #first_search 
        if counter == 1:
            # first try we just need the categories for one page
            new_categories = get_categories_for_page(page_id)
            # print(new_categories)
        else: 
            # now we loop over the results from before
            subcats_ids = get_subcats_ids_for_cats(old_categories) # new set of all the subcategories for the new level
            # print(subcats_ids)
            new_categories = set()
            # loop over new pages
            for x in subcats_ids:
                new_categories.update(get_categories_for_page(x))
            # print(new_categories)

        new_intersection = get_intersection_cat_target(new_categories, target_cats) 
        if new_intersection:
            for inter in new_intersection:
                intersection[inter] = intersection.get(inter, 0) + 1
        
        old_categories = new_categories - visited # to check in the next step

        if not old_categories: # categories gets empty
            break

        visited.update(new_categories)
        counter +=1

        if counter-1 >= 5 and intersection: # -1 because i'm incrementing the counter earlier and it triggers a layer early
            max_count = max(intersection.values())
            top = {k for k, v in intersection.items() if v == max_count}
            
            if len(top) == 1:
                UNIQUE_TOP = True 

    print(f"Total layers checked: {counter-1}")
    print(intersection)
    if not intersection:
        return set()
    else:
        return top

In [459]:
BFS(page_id_smartphone,target_cats)

{'Computer engineering', 'Computer science', 'Consumer electronics'}

In [460]:
BFS_hard(page_id_smartphone,target_cats)

Checking layer 5
current dictionary {'Consumer electronics': 3, 'Embedded systems': 2, 'Classes of computers': 1, 'Application software': 2, 'Computer engineering': 2, 'Computer science': 2, 'Manufacturing': 2, 'Electrical engineering': 1, 'Computer architecture': 1, 'Software': 1, 'Information systems': 1, 'Telecommunications': 1, 'Electronics manufacturing': 1, 'Mechanical engineering': 1}
Checking layer 10
current dictionary {'Consumer electronics': 3, 'Embedded systems': 2, 'Classes of computers': 3, 'Application software': 2, 'Computer engineering': 4, 'Computer science': 4, 'Manufacturing': 3, 'Electrical engineering': 5, 'Computer architecture': 3, 'Software': 2, 'Information systems': 3, 'Telecommunications': 5, 'Electronics manufacturing': 1, 'Mechanical engineering': 3, 'Software engineering': 3, 'Information technology': 3, 'Digital media': 4, 'Systems engineering': 2, 'Data': 2, 'Management': 1, 'Control theory': 3, 'Human–computer interaction': 3, 'Tools': 3, 'Automation':

{'Computer science', 'Electrical engineering', 'Telecommunications'}

In [461]:
BFS(page_id_nn,target_cats)

{'Biological engineering', 'Biotechnology', 'Medicine', 'Software engineering'}

In [462]:
BFS_hard(page_id_nn,target_cats)

Checking layer 5
current dictionary {'Biological engineering': 2, 'Biotechnology': 1, 'Medical technology': 1, 'Medicine': 1, 'Artificial intelligence': 1, 'Software engineering': 1}
Total layers checked: 6
{'Biological engineering': 2, 'Biotechnology': 2, 'Medical technology': 1, 'Medicine': 2, 'Artificial intelligence': 1, 'Software engineering': 3, 'Information technology': 2, 'Data': 2, 'Control theory': 1, 'Computer engineering': 1, 'Computer science': 1, 'Systems engineering': 2, 'Electrical engineering': 1, 'Management': 1, 'Information systems': 1, 'Manufacturing': 1}


{'Software engineering'}

In [463]:
page_id_gpu = 390214
BFS(page_id_gpu,target_cats)

{'Computer architecture',
 'Computer engineering',
 'Digital electronics',
 'Digital media',
 'Human–computer interaction',
 'Integrated circuits'}

In [464]:
BFS_hard(page_id_gpu,target_cats)

Checking layer 5
current dictionary {'Electronic design': 2, 'Artificial intelligence': 1, 'Digital electronics': 3, 'Integrated circuits': 3, 'Computer engineering': 3, 'Human–computer interaction': 3, 'Electrical engineering': 1, 'Digital media': 2, 'Computer architecture': 2, 'Consumer electronics': 1, 'Information systems': 1, 'Computer science': 1, 'Electrical components': 1, 'Information technology': 1, 'Application software': 1, 'Multimedia': 1, 'Machines': 1, 'Tools': 1, 'Manufacturing': 1, 'Systems engineering': 1, 'Semiconductors': 1}
Total layers checked: 9
{'Electronic design': 2, 'Artificial intelligence': 1, 'Digital electronics': 3, 'Integrated circuits': 3, 'Computer engineering': 4, 'Human–computer interaction': 3, 'Electrical engineering': 5, 'Digital media': 4, 'Computer architecture': 4, 'Consumer electronics': 1, 'Information systems': 3, 'Computer science': 3, 'Electrical components': 4, 'Information technology': 4, 'Application software': 2, 'Multimedia': 1, 'Mac

{'Electrical engineering'}

In [465]:
# transistors 30011
page_id_transistor = 30011

In [466]:
BFS(page_id_transistor,target_cats)

{'Electrical components', 'Electrical engineering'}

In [467]:
BFS_hard(page_id_transistor,target_cats)

Checking layer 5
current dictionary {'Electrical components': 3, 'Electrical engineering': 3, 'Semiconductors': 1, 'Manufacturing': 2, 'Materials science': 1, 'Energy': 1, 'Computer engineering': 1, 'Mechanical engineering': 1, 'Computer science': 1, 'Systems engineering': 1}
Total layers checked: 8
{'Electrical components': 5, 'Electrical engineering': 6, 'Semiconductors': 1, 'Manufacturing': 4, 'Materials science': 1, 'Energy': 1, 'Computer engineering': 4, 'Mechanical engineering': 4, 'Computer science': 3, 'Systems engineering': 1, 'Telecommunications': 4, 'Machines': 2, 'Tools': 1, 'Civil engineering': 2, 'Construction': 2, 'Electronics manufacturing': 1, 'Companies': 3, 'Architecture': 2, 'Information technology': 1, 'Computer architecture': 1, 'Classes of computers': 1, 'Data': 1, 'Management': 1}


{'Electrical engineering'}

In [468]:
page_id_ml = 233488

In [469]:
BFS(page_id_ml,target_cats)

{'Information technology'}

In [470]:
BFS_hard(page_id_ml,target_cats)

Checking layer 5
current dictionary {'Artificial intelligence': 1, 'Software engineering': 1, 'Control theory': 1, 'Data': 1, 'Information technology': 1, 'Systems engineering': 1, 'Computer engineering': 1}
Total layers checked: 5
{'Artificial intelligence': 1, 'Software engineering': 1, 'Control theory': 1, 'Data': 1, 'Information technology': 2, 'Systems engineering': 1, 'Computer engineering': 1, 'Electrical engineering': 1, 'Computer science': 1}


{'Information technology'}

In [471]:
page_dna_fingerprints = 44290

In [472]:
BFS(page_dna_fingerprints,target_cats)

{'Biotechnology',
 'Forensic science',
 'Human–computer interaction',
 'Information technology',
 'Medicine',
 'Tools'}

In [473]:
BFS_hard(page_dna_fingerprints,target_cats)

Checking layer 5
current dictionary {'Forensic science': 2, 'Data': 1, 'Human–computer interaction': 1, 'Information technology': 1, 'Medicine': 1, 'Biotechnology': 1, 'Tools': 1, 'Biological engineering': 1, 'Systems engineering': 1}
Total layers checked: 9
{'Forensic science': 2, 'Data': 4, 'Human–computer interaction': 3, 'Information technology': 5, 'Medicine': 4, 'Biotechnology': 3, 'Tools': 4, 'Biological engineering': 2, 'Systems engineering': 3, 'Software engineering': 4, 'Artificial intelligence': 2, 'Computer science': 4, 'Machines': 1, 'Computer architecture': 3, 'Information systems': 2, 'Application software': 1, 'Materials science': 2, 'Telecommunications': 3, 'Mechanical engineering': 2, 'Computer security': 1, 'Computer engineering': 4, 'Manufacturing': 2, 'Military science': 2, 'Electrical engineering': 3, 'Digital media': 2, 'Architecture': 3, 'Software': 1, 'Management': 2, 'Multimedia': 1, 'Construction': 1, 'Energy': 1, 'Operating systems': 1, 'Internet': 1, 'Medic

{'Information technology'}